# 📊 Model Selection in Climatology Engine

This notebook introduces criteria for selecting the best statistical model.

**What you will learn:**
- AIC (Akaike Information Criterion)
- AICc (corrected for small sample sizes)
- BIC (Bayesian Information Criterion)
- Likelihood Ratio Test
- Interpreting ΔAICc and Akaike Weight
- Selecting the best model with multiple criteria

---

## 📐 Model Selection Theory

### 1. Akaike Information Criterion (AIC)

$$
AIC = 2k - 2\ln(\hat{L})
$$

where:
- $k$: number of model parameters
- $\hat{L}$: maximum likelihood

### 2. Corrected AIC (AICc)

$$
AIC_c = AIC + \frac{2k(k+1)}{n-k-1}
$$

where $n$ is the sample size. Recommended for small samples ($n/k < 40$).

### 3. Bayesian Information Criterion (BIC)

$$
BIC = k\ln(n) - 2\ln(\hat{L})
$$

### 4. Interpreting ΔAICc

| ΔAICc | Interpretation |
|-------|----------------|
| ≤ 2 | Strong support |
| 4-7 | Weak support |
| > 10 | No support |

### 5. Akaike Weight

$$
w_i = \frac{\exp(-\Delta_i / 2)}{\sum_{j=1}^{m} \exp(-\Delta_j / 2)}
$$

where $\Delta_i = AICc_i - AICc_{min}$.

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from core.engine.plugin_loader import load_plugins

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ Libraries loaded.')

In [ ]:
# Load distribution plugins
plugins = load_plugins()
print(f'✅ Number of loaded distributions: {len(plugins)}')

for code, dist in plugins.items():
    print(f"   [{code}] {dist.name} (params: {dist.params})")

distributions = {dist.name: dist for dist in plugins.values()}

In [ ]:
# Load sample data
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values

# Select tmean data for one year
data_year = data[:365, 1]

print(f'📊 Number of samples: {len(data_year)}')
print(f'   Mean: {np.mean(data_year):.2f}°C')
print(f'   Standard deviation: {np.std(data_year):.2f}°C')
print(f'   Sample size (n): {len(data_year)}')

In [ ]:
# Fit all distributions
results_all = {}
print("\n🔄 Fitting distributions...\n")

for name, dist in distributions.items():
    try:
        res = dist.fit(data_year)
        results_all[name] = res
        print(f"✅ {name}: AICc = {res.get('aicc', np.nan):.2f}, "
              f"BIC = {res.get('bic', np.nan):.2f}, "
              f"LogLik = {res.get('loglik', np.nan):.2f}")
    except Exception as e:
        print(f"❌ {name}: Error - {str(e)}")
        results_all[name] = None

print("\n✅ Fitting complete.")

In [ ]:
# Extract valid results
valid_results = {k: v for k, v in results_all.items() 
                  if v is not None and 'aicc' in v and not np.isnan(v['aicc'])}

print(f"✅ Number of valid models: {len(valid_results)}")
print(f"   Models: {list(valid_results.keys())}")

In [ ]:
# Calculate model selection metrics
def calculate_model_selection_metrics(results):
    """
    Calculate model selection metrics including:
    - AICc
    - ΔAICc
    - Akaike Weight
    - BIC
    - ΔBIC
    - BIC Weight
    """
    metrics = []
    
    # Find best AICc and BIC
    min_aicc = min(r['aicc'] for r in results.values() if 'aicc' in r)
    min_bic = min(r['bic'] for r in results.values() if 'bic' in r and not np.isnan(r['bic']))
    
    for name, res in results.items():
        aicc = res.get('aicc', np.nan)
        bic = res.get('bic', np.nan)
        loglik = res.get('loglik', np.nan)
        n_params = res.get('n_params', np.nan)
        
        delta_aicc = aicc - min_aicc if not np.isnan(aicc) else np.nan
        delta_bic = bic - min_bic if not np.isnan(bic) else np.nan
        
        metrics.append({
            'Model': name,
            'AICc': aicc,
            'ΔAICc': delta_aicc,
            'BIC': bic,
            'ΔBIC': delta_bic,
            'LogLik': loglik,
            'N_Params': n_params
        })
    
    # Calculate Akaike weights
    exp_vals = np.exp(-np.array([m['ΔAICc'] for m in metrics]) / 2)
    sum_exp = np.sum(exp_vals)
    for i, m in enumerate(metrics):
        m['AICc_Weight'] = exp_vals[i] / sum_exp
    
    # Calculate BIC weights
    exp_vals_bic = np.exp(-np.array([m['ΔBIC'] for m in metrics]) / 2)
    sum_exp_bic = np.sum(exp_vals_bic)
    for i, m in enumerate(metrics):
        m['BIC_Weight'] = exp_vals_bic[i] / sum_exp_bic
    
    return pd.DataFrame(metrics).sort_values('AICc').reset_index(drop=True)

selection_df = calculate_model_selection_metrics(valid_results)
selection_df.index = selection_df.index + 1

# Display formatted table
print("📊 Model Selection Metrics Table:")
print("=" * 100)
selection_df.round(4)

In [ ]:
# Display best model by AICc
best_aicc = selection_df.loc[selection_df['AICc'].idxmin()]
print("🏆 Best model by AICc:")
print("=" * 50)
print(f"   Model: {best_aicc['Model']}")
print(f"   AICc: {best_aicc['AICc']:.4f}")
print(f"   AICc Weight: {best_aicc['AICc_Weight']:.4f} ({best_aicc['AICc_Weight']*100:.1f}%)")
print(f"   N_Params: {best_aicc['N_Params']}")
print("=" * 50)

# Display best model by BIC
best_bic = selection_df.loc[selection_df['BIC'].idxmin()]
print("\n🏆 Best model by BIC:")
print("=" * 50)
print(f"   Model: {best_bic['Model']}")
print(f"   BIC: {best_bic['BIC']:.4f}")
print(f"   BIC Weight: {best_bic['BIC_Weight']:.4f} ({best_bic['BIC_Weight']*100:.1f}%)")
print(f"   N_Params: {best_bic['N_Params']}")
print("=" * 50)

In [ ]:
# Plot AICc weights
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2ecc71' if i == 0 else '#e74c3c' if i == 1 else '#3498db' 
          for i in range(len(selection_df))]

bars = ax.bar(selection_df['Model'], selection_df['AICc_Weight'], 
              color=colors, alpha=0.7, edgecolor='black', linewidth=1)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('AICc Weight', fontsize=12)
ax.set_title('Akaike Weights of Different Models', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, weight in zip(bars, selection_df['AICc_Weight']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{weight:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Plot ΔAICc comparison
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2ecc71' if d == 0 else '#e74c3c' if d > 10 else '#f39c12' 
          for d in selection_df['ΔAICc']]

bars = ax.bar(selection_df['Model'], selection_df['ΔAICc'], 
              color=colors, alpha=0.7, edgecolor='black', linewidth=1)

ax.axhline(y=2, color='green', linestyle='--', linewidth=2, alpha=0.7, label='ΔAICc = 2 (Strong support)')
ax.axhline(y=7, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='ΔAICc = 7 (Weak support)')
ax.axhline(y=10, color='red', linestyle='--', linewidth=2, alpha=0.7, label='ΔAICc = 10 (No support)')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('ΔAICc', fontsize=12)
ax.set_title('ΔAICc Comparison of Models', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

for bar, delta in zip(bars, selection_df['ΔAICc']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            f'{delta:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ΔAICc interpretation table
def interpret_delta_aicc(delta):
    """Interpret ΔAICc based on Burnham & Anderson guidelines"""
    if delta <= 2:
        return 'Strong support', 'green'
    elif delta <= 4:
        return 'Moderate support', 'lightgreen'
    elif delta <= 7:
        return 'Weak support', 'orange'
    elif delta <= 10:
        return 'Very weak support', 'darkorange'
    else:
        return 'No support', 'red'

interpretation_df = selection_df[['Model', 'ΔAICc', 'AICc_Weight']].copy()
interpretation_df['ΔAICc_Interpretation'], interpretation_df['Color'] = zip(*interpretation_df['ΔAICc'].apply(interpret_delta_aicc))

print("📋 ΔAICc Interpretation:")
print("=" * 80)
interpretation_df[['Model', 'ΔAICc', 'AICc_Weight', 'ΔAICc_Interpretation']]

In [ ]:
# Compare AICc vs BIC
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(selection_df['Model']))
width = 0.35

bars1 = ax.bar(x - width/2, selection_df['AICc'], width, 
               label='AICc', color='#3498db', alpha=0.7, edgecolor='black', linewidth=1)
bars2 = ax.bar(x + width/2, selection_df['BIC'], width,
               label='BIC', color='#e74c3c', alpha=0.7, edgecolor='black', linewidth=1)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Criterion Value', fontsize=12)
ax.set_title('Comparison of AICc and BIC for Different Models', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(selection_df['Model'])
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.5, f'{height:.1f}',
            ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.5, f'{height:.1f}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Function to select best model by different criteria
def select_best_model_by_criterion(df, criterion='AICc'):
    """
    Select best model based on specified criterion
    criterion: 'AICc', 'BIC', 'LogLik', 'AICc_Weight', 'BIC_Weight'
    """
    if criterion in ['AICc', 'BIC']:
        best = df.loc[df[criterion].idxmin()]
    elif criterion in ['AICc_Weight', 'BIC_Weight']:
        best = df.loc[df[criterion].idxmax()]
    elif criterion == 'LogLik':
        best = df.loc[df[criterion].idxmax()]
    else:
        raise ValueError(f"Invalid criterion: {criterion}")
    return best

print("\n📊 Best model selection by different criteria:")
print("=" * 60)

criteria = ['AICc', 'BIC', 'LogLik', 'AICc_Weight', 'BIC_Weight']
for crit in criteria:
    try:
        best = select_best_model_by_criterion(selection_df, crit)
        print(f"\n✅ By {crit}:")
        print(f"   Best model: {best['Model']}")
        print(f"   Value: {best[crit]:.4f}")
    except Exception as e:
        print(f"\n⚠️ Error calculating {crit}: {str(e)}")

print("\n" + "=" * 60)

## 📋 Summary

In this notebook you learned:

✅ Model selection criteria: AIC, AICc, BIC
✅ Calculating and interpreting ΔAICc
✅ Calculating Akaike Weights
✅ Comparing models with different criteria
✅ Selecting the best model with multiple criteria

---

**Key Takeaways:**

1. **AICc** is more suitable than AIC for small samples.
2. **Akaike Weight** represents the probability that a model is the best.
3. **ΔAICc < 2** indicates strong support for the model.
4. Models with **ΔAICc > 10** are practically not supported.
5. **BIC** favors simpler models compared to AICc.

---

**Next Steps:**
- Notebook 05: Quality Control (Quality Flag)
- Notebook 06: Bootstrap Uncertainty
- Notebook 07: Parallel Processing